# Notebook 06 — XAI Gate #2 (Post-Tuning)
**Project:** Customer Churn Prediction  
**Depends on:** `02_preprocessing`, `03_baseline`, `04_xai_gate1`, `05_tuning`  
**Output:** `xai_gate2_results.json` + SHAP plots per kandidat + verdict lulus/gagal per model

Pipeline:
1. Load tuned models + splits
2. Evaluasi D1–D4 untuk semua 6 kandidat individual
3. SHAP penuh untuk Voting Ensemble (komposit TreeExplainer)
4. Verdict final + simpan ke WandB

---
## Install & Import

In [1]:
!pip install shap lightgbm xgboost wandb --quiet
print('OK install selesai.')

OK install selesai.


In [2]:
import os, json, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import joblib
import shap
from sklearn.inspection import permutation_importance
from sklearn.metrics    import average_precision_score
import wandb

print('OK import selesai.')

OK import selesai.


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret('WANDB')
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ardiyanto24 (ardiyanto24-indonesian-national-police) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

---
## Konstanta Global

In [4]:
# ── Path ───────────────────────────────────────────────────────────────────────
PREPROCESSING_DIR = '/kaggle/input/notebooks/ardiyanto24/tccp-preprocessing-v2/artifacts'
TUNING_DIR        = '/kaggle/input/notebooks/ardiyanto24/tccp-hyperparameter-tuning/artifacts'
SPLITS_PATH       = f'{PREPROCESSING_DIR}/splits.joblib'
OUTPUT_DIR        = '/kaggle/working/artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── WandB ──────────────────────────────────────────────────────────────────────
WANDB_PROJECT      = 'customer-churn-prediction'
WANDB_GATE2_GROUP  = 'xai-gate2-06'

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── XAI Gate kriteria (dari config/settings.py) ────────────────────────────────
# D1 — konsistensi SHAP: minimal 50% dari EXPECTED masuk top-10 SHAP
# D2 — konsistensi Permutation Importance: minimal 50% dari EXPECTED masuk top-10 PI
# D3 — tidak ada fitur 'garbage' mendominasi top-5 SHAP
# D4 — konsistensi arah SHAP: fitur kunci searah dengan domain knowledge
EXPECTED_IMPORTANT_FEATURES = [
    'Contract',
    'tenure',
    'MonthlyCharges',
    'tc_residual',          # TotalCharges diganti tc_residual setelah feature engineering
    'InternetService',
]
XAI_TOP_N      = 10       # evaluasi top-N features
XAI_MIN_OVERLAP = 0.5     # minimal 50% expected harus muncul

# Fitur yang TIDAK boleh mendominasi top-5 (D3 — garbage check)
# Fitur dengan korelasi rendah atau tidak ada domain justification
GARBAGE_FEATURES = ['customerID', 'id']

# Arah SHAP yang diharapkan (D4) — berdasarkan domain knowledge churn:
# nilai tinggi → kontribusi positif ke churn (arah +)
# nilai rendah → kontribusi positif ke churn (arah -)
EXPECTED_DIRECTION = {
    'MonthlyCharges': '+',   # tagihan tinggi → churn lebih mungkin
    'tenure'        : '-',   # tenure panjang → churn lebih kecil
    'tc_residual'   : '+',   # residual tinggi → anomali biaya → churn lebih mungkin
}

# ── Kandidat (sama dengan Notebook 05) ────────────────────────────────────────
CANDIDATES = [
    {'model_name': 'lightgbm',            'balance': 'class_weight', 'tuned_pr_auc': 0.7522},
    {'model_name': 'xgboost',             'balance': 'class_weight', 'tuned_pr_auc': 0.7521},
    {'model_name': 'xgboost',             'balance': 'smote',        'tuned_pr_auc': 0.7485},
    {'model_name': 'lightgbm',            'balance': 'smote',        'tuned_pr_auc': 0.7477},
    {'model_name': 'logistic_regression', 'balance': 'class_weight', 'tuned_pr_auc': 0.7410},
    {'model_name': 'logistic_regression', 'balance': 'smote',        'tuned_pr_auc': 0.7406},
]

# Sample size untuk SHAP (komputasi mahal di val set besar)
SHAP_SAMPLE_SIZE       = 5000
SHAP_SAMPLE_SIZE_ENSEMBLE = 2000   # lebih kecil karena 3 model

# PI sample size
PI_SAMPLE_SIZE = 5000
PI_N_REPEATS   = 5

print('OK konstanta global terdefinisi.')
print(f'   EXPECTED_IMPORTANT_FEATURES : {EXPECTED_IMPORTANT_FEATURES}')
print(f'   XAI_TOP_N                   : {XAI_TOP_N}')
print(f'   XAI_MIN_OVERLAP             : {XAI_MIN_OVERLAP}')
print(f'   Jumlah kandidat             : {len(CANDIDATES)}')

OK konstanta global terdefinisi.
   EXPECTED_IMPORTANT_FEATURES : ['Contract', 'tenure', 'MonthlyCharges', 'tc_residual', 'InternetService']
   XAI_TOP_N                   : 10
   XAI_MIN_OVERLAP             : 0.5
   Jumlah kandidat             : 6


---
## Load Artifacts

In [5]:
# ── Load splits ────────────────────────────────────────────────────────────────
print('Loading splits...')
splits        = joblib.load(SPLITS_PATH)
X_train       = splits['X_train']
y_train       = splits['y_train']
X_val         = splits['X_val']
y_val         = splits['y_val']
feature_names = splits['feature_names']

print(f'  X_train shape : {X_train.shape}')
print(f'  X_val   shape : {X_val.shape}')
print(f'  Feature names : {len(feature_names)} fitur')

# Subsample untuk SHAP dan PI (reproducible)
rng = np.random.default_rng(RANDOM_SEED)
shap_idx  = rng.choice(len(X_val), size=min(SHAP_SAMPLE_SIZE, len(X_val)), replace=False)
pi_idx    = rng.choice(len(X_val), size=min(PI_SAMPLE_SIZE,   len(X_val)), replace=False)
ens_idx   = rng.choice(len(X_val), size=min(SHAP_SAMPLE_SIZE_ENSEMBLE, len(X_val)), replace=False)

X_shap  = X_val[shap_idx]  if hasattr(X_val, '__getitem__') else X_val.iloc[shap_idx]
y_shap  = y_val[shap_idx]  if hasattr(y_val, '__getitem__') else y_val.iloc[shap_idx]
X_pi    = X_val[pi_idx]    if hasattr(X_val, '__getitem__') else X_val.iloc[pi_idx]
y_pi    = y_val[pi_idx]    if hasattr(y_val, '__getitem__') else y_val.iloc[pi_idx]
X_ens   = X_val[ens_idx]   if hasattr(X_val, '__getitem__') else X_val.iloc[ens_idx]
y_ens   = y_val[ens_idx]   if hasattr(y_val, '__getitem__') else y_val.iloc[ens_idx]

# Konversi ke DataFrame jika numpy array (untuk SHAP feature names)
if not isinstance(X_shap, pd.DataFrame):
    X_shap = pd.DataFrame(X_shap, columns=feature_names)
if not isinstance(X_pi, pd.DataFrame):
    X_pi   = pd.DataFrame(X_pi,   columns=feature_names)
if not isinstance(X_ens, pd.DataFrame):
    X_ens  = pd.DataFrame(X_ens,  columns=feature_names)
if not isinstance(X_val, pd.DataFrame):
    X_val_df = pd.DataFrame(X_val, columns=feature_names)
else:
    X_val_df = X_val

print(f'  SHAP sample   : {len(X_shap)} baris')
print(f'  PI sample     : {len(X_pi)} baris')
print(f'  Ensemble SHAP : {len(X_ens)} baris')

Loading splits...
  X_train shape : (415935, 29)
  X_val   shape : (89129, 29)
  Feature names : 29 fitur
  SHAP sample   : 5000 baris
  PI sample     : 5000 baris
  Ensemble SHAP : 2000 baris


In [6]:
# ── Load tuned models ──────────────────────────────────────────────────────────
print('Loading tuned models...')
tuned_models = {}

for cand in CANDIDATES:
    key   = f"{cand['model_name']}__{cand['balance']}"
    fpath = os.path.join(TUNING_DIR, f'tuned_{key}.joblib')
    if os.path.exists(fpath):
        tuned_models[key] = joblib.load(fpath)
        print(f'  OK {key}')
    else:
        print(f'  NOT FOUND: {fpath}')

# Load voting ensemble
ens_path = os.path.join(TUNING_DIR, 'tuned_voting_ensemble.joblib')
voting_ensemble = joblib.load(ens_path) if os.path.exists(ens_path) else None
print(f'  OK voting_ensemble' if voting_ensemble else '  NOT FOUND: voting_ensemble')

print(f'\nTotal individual loaded: {len(tuned_models)}/{len(CANDIDATES)}')

Loading tuned models...
  OK lightgbm__class_weight
  OK xgboost__class_weight
  OK xgboost__smote
  OK lightgbm__smote
  OK logistic_regression__class_weight
  OK logistic_regression__smote
  OK voting_ensemble

Total individual loaded: 6/6


---
## Deklarasi Class & Function

> Semua class stateless. Menerima model + data, mengembalikan hasil dict.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: SHAPAnalyzer
# Tanggung jawab: hitung SHAP values + summary plot untuk satu model
# Mendukung tree-based (TreeExplainer) dan linear (LinearExplainer)
# ══════════════════════════════════════════════════════════════════════════════
class SHAPAnalyzer:
    """
    Hitung SHAP values dan buat summary plot untuk satu model tuned.
    Output: shap_values (array), top_N_features (list), plot disimpan ke disk.
    """

    def __init__(self, model_key: str, model, X_sample: pd.DataFrame,
                 feature_names: list, top_n: int = XAI_TOP_N):
        self.model_key     = model_key
        self.model         = model
        self.X_sample      = X_sample
        self.feature_names = feature_names
        self.top_n         = top_n

    def _get_explainer(self):
        """Pilih explainer yang sesuai berdasarkan tipe model."""
        name = self.model_key.split('__')[0]
        if name in ('lightgbm', 'xgboost'):
            return shap.TreeExplainer(self.model)
        elif name == 'logistic_regression':
            # LinearExplainer butuh masker
            masker = shap.maskers.Independent(self.X_sample, max_samples=500)
            return shap.LinearExplainer(self.model, masker)
        else:
            raise ValueError(f'Tipe model tidak dikenal: {name}')

    def run(self) -> dict:
        SEP = '=' * 65
        print(f'\n{SEP}')
        print(f'  SHAP: {self.model_key}')
        print(SEP)
        t_start = time.time()

        explainer    = self._get_explainer()
        shap_values  = explainer(self.X_sample)

        # Ambil SHAP values untuk kelas positif (churn=1)
        if hasattr(shap_values, 'values'):
            sv = shap_values.values
        else:
            sv = shap_values

        # Jika 3-dim (n_samples, n_features, n_classes) ambil kelas 1
        if sv.ndim == 3:
            sv = sv[:, :, 1]

        # Mean absolute SHAP per fitur
        mean_abs_shap = np.abs(sv).mean(axis=0)
        shap_df = pd.DataFrame({
            'feature'   : self.feature_names,
            'mean_abs_shap': mean_abs_shap,
        }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

        top_features = shap_df['feature'].head(self.top_n).tolist()

        print(f'  Top-{self.top_n} SHAP features:')
        for i, row in shap_df.head(self.top_n).iterrows():
            mark = '*' if row['feature'] in EXPECTED_IMPORTANT_FEATURES else ' '
            print(f'    {mark} {i+1:2d}. {row["feature"]:<35} {row["mean_abs_shap"]:.5f}')

        # Summary plot (bar)
        fig, ax = plt.subplots(figsize=(9, 6))
        shap_df_plot = shap_df.head(15).sort_values('mean_abs_shap')
        colors = ['#1D9E75' if f in EXPECTED_IMPORTANT_FEATURES else '#B4B2A9'
                  for f in shap_df_plot['feature']]
        ax.barh(shap_df_plot['feature'], shap_df_plot['mean_abs_shap'], color=colors)
        ax.set_xlabel('Mean |SHAP value|', fontsize=11)
        ax.set_title(f'SHAP Feature Importance — {self.model_key}', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

        # Legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#1D9E75', label='Expected important'),
            Patch(facecolor='#B4B2A9', label='Other'),
        ]
        ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
        plt.tight_layout()

        plot_path = os.path.join(OUTPUT_DIR, f'shap_{self.model_key}.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()

        elapsed = time.time() - t_start
        print(f'  Plot disimpan : {plot_path}')
        print(f'  Waktu         : {elapsed:.1f} detik')

        return {
            'top_features'  : top_features,
            'shap_df'       : shap_df,
            'shap_values_sv': sv,
            'plot_path'     : plot_path,
        }

print('OK SHAPAnalyzer terdefinisi.')

OK SHAPAnalyzer terdefinisi.


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: PermutationAnalyzer
# Tanggung jawab: hitung Permutation Importance (D2)
# ══════════════════════════════════════════════════════════════════════════════
class PermutationAnalyzer:
    """
    Hitung Permutation Importance menggunakan sklearn.
    Scoring: average_precision (PR-AUC) agar konsisten dengan tuning.
    """

    def __init__(self, model_key: str, model,
                 X_sample: pd.DataFrame, y_sample,
                 feature_names: list, top_n: int = XAI_TOP_N):
        self.model_key     = model_key
        self.model         = model
        self.X_sample      = X_sample
        self.y_sample      = y_sample
        self.feature_names = feature_names
        self.top_n         = top_n

    def run(self) -> dict:
        print(f'  PI: {self.model_key} ... ', end='', flush=True)
        t_start = time.time()

        result = permutation_importance(
            self.model, self.X_sample, self.y_sample,
            n_repeats   = PI_N_REPEATS,
            random_state= RANDOM_SEED,
            scoring     = 'average_precision',
            n_jobs      = -1,
        )

        pi_df = pd.DataFrame({
            'feature'      : self.feature_names,
            'mean_decrease': result.importances_mean,
            'std'          : result.importances_std,
        }).sort_values('mean_decrease', ascending=False).reset_index(drop=True)

        top_features = pi_df['feature'].head(self.top_n).tolist()

        elapsed = time.time() - t_start
        print(f'selesai ({elapsed:.1f}s)')
        print(f'    Top-{self.top_n}: {top_features}')

        return {
            'top_features': top_features,
            'pi_df'       : pi_df,
        }

print('OK PermutationAnalyzer terdefinisi.')

OK PermutationAnalyzer terdefinisi.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: DirectionChecker
# Tanggung jawab: evaluasi D4 — arah SHAP konsisten dengan domain knowledge
# ══════════════════════════════════════════════════════════════════════════════
class DirectionChecker:
    """
    Validasi arah SHAP untuk fitur numerik kunci.

    Logika:
    - Untuk fitur dengan expected_direction '+': korelasi Pearson antara
      nilai fitur dan SHAP value harus positif
    - Untuk fitur dengan expected_direction '-': korelasi harus negatif

    Threshold: |korelasi| > 0.05 dianggap cukup untuk konfirmasi arah.
    Model lulus D4 jika semua fitur yang ada di top features mempunyai
    arah yang benar, atau minimal 2/3 dari EXPECTED_DIRECTION.
    """

    CORR_THRESHOLD = 0.05

    def __init__(self, model_key: str, X_sample: pd.DataFrame,
                 shap_values_sv: np.ndarray, feature_names: list):
        self.model_key      = model_key
        self.X_sample       = X_sample
        self.shap_values_sv = shap_values_sv
        self.feature_names  = feature_names

    def run(self) -> dict:
        results     = {}
        n_checked   = 0
        n_correct   = 0

        for feat, expected_dir in EXPECTED_DIRECTION.items():
            if feat not in self.feature_names:
                continue
            feat_idx  = list(self.feature_names).index(feat)
            feat_vals = self.X_sample.iloc[:, feat_idx].values if isinstance(
                self.X_sample, pd.DataFrame) else self.X_sample[:, feat_idx]
            shap_vals = self.shap_values_sv[:, feat_idx]

            corr = np.corrcoef(feat_vals.astype(float), shap_vals)[0, 1]
            if np.isnan(corr):
                corr = 0.0

            actual_dir = '+' if corr > 0 else '-'
            magnitude_ok = abs(corr) > self.CORR_THRESHOLD
            direction_ok = (actual_dir == expected_dir) and magnitude_ok

            n_checked += 1
            if direction_ok:
                n_correct += 1

            results[feat] = {
                'expected' : expected_dir,
                'actual'   : actual_dir,
                'corr'     : round(corr, 4),
                'pass'     : direction_ok,
            }
            status = 'OK' if direction_ok else 'FAIL'
            print(f'    [{status}] {feat:<30} expected={expected_dir}  '
                  f'actual={actual_dir}  corr={corr:+.4f}')

        d4_pass = (n_correct >= max(1, round(n_checked * 2 / 3)))
        print(f'    D4 result: {n_correct}/{n_checked} fitur arah benar '
              f'→ {"LULUS" if d4_pass else "GAGAL"}')

        return {
            'direction_results': results,
            'n_checked'        : n_checked,
            'n_correct'        : n_correct,
            'd4_pass'          : d4_pass,
        }

print('OK DirectionChecker terdefinisi.')

OK DirectionChecker terdefinisi.


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: XAIGate2Evaluator
# Tanggung jawab: orkestrasi D1–D4 untuk satu model, return verdict
# ══════════════════════════════════════════════════════════════════════════════
class XAIGate2Evaluator:
    """
    Menjalankan evaluasi D1–D4 untuk satu kandidat model:

    D1 — SHAP consistency:
         minimal XAI_MIN_OVERLAP dari EXPECTED_IMPORTANT_FEATURES
         muncul di top-XAI_TOP_N SHAP features

    D2 — Permutation Importance consistency:
         minimal XAI_MIN_OVERLAP dari EXPECTED_IMPORTANT_FEATURES
         muncul di top-XAI_TOP_N PI features

    D3 — Garbage feature check:
         tidak ada fitur dari GARBAGE_FEATURES yang masuk top-5 SHAP

    D4 — Direction check:
         arah SHAP fitur numerik kunci konsisten dengan domain knowledge
         (setidaknya 2/3 dari EXPECTED_DIRECTION)

    Verdict: LULUS jika semua D1–D4 lulus.
    """

    def __init__(self, model_key: str, model,
                 X_shap: pd.DataFrame, y_shap,
                 X_pi: pd.DataFrame,   y_pi,
                 feature_names: list):
        self.model_key     = model_key
        self.model         = model
        self.X_shap        = X_shap
        self.y_shap        = y_shap
        self.X_pi          = X_pi
        self.y_pi          = y_pi
        self.feature_names = feature_names

    def _evaluate_overlap(self, top_features: list, label: str) -> dict:
        """
        Hitung overlap antara top_features dan EXPECTED.

        Menggunakan prefix-based matching (Opsi B) agar robust terhadap OHE:
        - Exact match: 'tenure' cocok dengan 'tenure'
        - Prefix match: 'Contract' cocok dengan 'Contract_Two year', 'Contract_One year'
        - Prefix match: 'InternetService' cocok dengan 'InternetService_Fiber optic'

        Dengan cara ini EXPECTED_IMPORTANT_FEATURES tetap mendefinisikan konsep bisnis
        (nama pre-OHE), bukan nama kolom teknis post-encoding.
        """
        found = []
        matched_by = {}  # untuk logging: expected → actual feature yang match
        for expected in EXPECTED_IMPORTANT_FEATURES:
            matched = [
                f for f in top_features
                if f == expected or f.startswith(expected + '_')
            ]
            if matched:
                found.append(expected)
                matched_by[expected] = matched
        overlap  = len(found) / len(EXPECTED_IMPORTANT_FEATURES)
        passes   = overlap >= XAI_MIN_OVERLAP
        print(f'  {label} overlap: {len(found)}/{len(EXPECTED_IMPORTANT_FEATURES)} '
              f'({overlap:.0%}) — {"LULUS" if passes else "GAGAL"}')
        for exp in EXPECTED_IMPORTANT_FEATURES:
            if exp in matched_by:
                print(f'    [OK] {exp:<25} via {matched_by[exp]}')
            else:
                print(f'    [--] {exp:<25} tidak ditemukan di top-{XAI_TOP_N}')
        return {'found': found, 'overlap': overlap, 'pass': passes,
                'matched_by': matched_by}

    def _evaluate_garbage(self, top5_features: list) -> bool:
        """D3: cek tidak ada garbage feature di top-5."""
        garbage_found = [f for f in GARBAGE_FEATURES if f in top5_features]
        passes        = len(garbage_found) == 0
        if garbage_found:
            print(f'  D3 GAGAL: garbage features di top-5: {garbage_found}')
        else:
            print(f'  D3 LULUS: tidak ada garbage feature di top-5')
        return passes

    def run(self) -> dict:
        SEP = '=' * 65
        print(f'\n{SEP}')
        print(f'  XAI Gate #2: {self.model_key}')
        print(SEP)

        # ── D1: SHAP ──────────────────────────────────────────────────────────
        print('\n  [D1] SHAP consistency')
        shap_result = SHAPAnalyzer(
            model_key     = self.model_key,
            model         = self.model,
            X_sample      = self.X_shap,
            feature_names = self.feature_names,
        ).run()
        d1_result = self._evaluate_overlap(shap_result['top_features'], 'D1 SHAP')

        # ── D2: Permutation Importance ────────────────────────────────────────
        print('\n  [D2] Permutation Importance consistency')
        pi_result = PermutationAnalyzer(
            model_key     = self.model_key,
            model         = self.model,
            X_sample      = self.X_pi,
            y_sample      = self.y_pi,
            feature_names = self.feature_names,
        ).run()
        d2_result = self._evaluate_overlap(pi_result['top_features'], 'D2 PI')

        # ── D3: Garbage check ─────────────────────────────────────────────────
        print('\n  [D3] Garbage feature check')
        top5 = shap_result['top_features'][:5]
        d3_pass = self._evaluate_garbage(top5)

        # ── D4: Direction check ───────────────────────────────────────────────
        print('\n  [D4] Direction check')
        d4_result = DirectionChecker(
            model_key      = self.model_key,
            X_sample       = self.X_shap,
            shap_values_sv = shap_result['shap_values_sv'],
            feature_names  = self.feature_names,
        ).run()

        # ── Verdict ───────────────────────────────────────────────────────────
        d1_pass      = d1_result['pass']
        d2_pass      = d2_result['pass']
        d3_pass_flag = d3_pass
        d4_pass      = d4_result['d4_pass']
        all_pass     = d1_pass and d2_pass and d3_pass_flag and d4_pass

        print(f'\n  ── VERDICT: {self.model_key} ──')
        print(f'  D1 (SHAP)         : {"LULUS" if d1_pass else "GAGAL"}')
        print(f'  D2 (PI)           : {"LULUS" if d2_pass else "GAGAL"}')
        print(f'  D3 (Garbage)      : {"LULUS" if d3_pass_flag else "GAGAL"}')
        print(f'  D4 (Direction)    : {"LULUS" if d4_pass else "GAGAL"}')
        print(f'  FINAL             : {"✓ LULUS" if all_pass else "✗ GAGAL"}')

        return {
            'key'          : self.model_key,
            'd1_pass'      : d1_pass,
            'd1_overlap'   : d1_result['overlap'],
            'd1_found'     : d1_result['found'],
            'd2_pass'      : d2_pass,
            'd2_overlap'   : d2_result['overlap'],
            'd2_found'     : d2_result['found'],
            'd3_pass'      : d3_pass_flag,
            'd4_pass'      : d4_pass,
            'd4_details'   : d4_result['direction_results'],
            'all_pass'     : all_pass,
            'shap_top10'   : shap_result['top_features'],
            'pi_top10'     : pi_result['top_features'],
            'shap_plot'    : shap_result['plot_path'],
        }

print('OK XAIGate2Evaluator terdefinisi.')

OK XAIGate2Evaluator terdefinisi.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: EnsembleSHAPAnalyzer
# Tanggung jawab: SHAP komposit untuk Voting Ensemble
# Strategi: rata-rata SHAP dari ketiga komponen (weighted by ensemble weights)
# ══════════════════════════════════════════════════════════════════════════════
class EnsembleSHAPAnalyzer:
    """
    SHAP untuk Voting Ensemble tidak bisa langsung menggunakan TreeExplainer
    karena VotingClassifier bukan tree tunggal.

    Strategi: hitung SHAP per komponen dengan bobotnya masing-masing,
    lalu rata-ratakan (weighted average).

    Komponen diambil dari voting_ensemble.estimators_ — urutan sesuai
    dengan yang difit di Notebook 05.
    """

    def __init__(self, ensemble, X_sample: pd.DataFrame,
                 feature_names: list, top_n: int = XAI_TOP_N):
        self.ensemble      = ensemble
        self.X_sample      = X_sample
        self.feature_names = feature_names
        self.top_n         = top_n

    def run(self) -> dict:
        SEP = '=' * 65
        print(f'\n{SEP}')
        print(f'  SHAP Ensemble (komposit)')
        print(SEP)
        t_start = time.time()

        # estimators_ = list of fitted models (post-fit)
        # estimators  = list of (name, estimator) tuples (pre-fit, untuk nama)
        fitted_models   = self.ensemble.estimators_
        named_estimators = self.ensemble.estimators   # list of (name, estimator)
        weights         = self.ensemble.weights if self.ensemble.weights else [1] * len(fitted_models)
        total_w         = sum(weights)

        comp_names = [name for name, _ in named_estimators]
        print(f'  Komponen: {comp_names}')
        print(f'  Bobot   : {weights}')

        weighted_shap = np.zeros((len(self.X_sample), len(self.feature_names)))

        for (name, _), model, w in zip(named_estimators, fitted_models, weights):
            print(f'  Menghitung SHAP untuk {name} (bobot={w})...', flush=True)
            explainer   = shap.TreeExplainer(model)
            sv          = explainer(self.X_sample)
            sv_arr      = sv.values if hasattr(sv, 'values') else sv
            if sv_arr.ndim == 3:
                sv_arr = sv_arr[:, :, 1]
            weighted_shap += (w / total_w) * sv_arr

        mean_abs_shap = np.abs(weighted_shap).mean(axis=0)
        shap_df = pd.DataFrame({
            'feature'      : self.feature_names,
            'mean_abs_shap': mean_abs_shap,
        }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

        top_features = shap_df['feature'].head(self.top_n).tolist()

        print(f'\n  Top-{self.top_n} SHAP Ensemble:')
        for i, row in shap_df.head(self.top_n).iterrows():
            mark = '*' if row['feature'] in EXPECTED_IMPORTANT_FEATURES else ' '
            print(f'    {mark} {i+1:2d}. {row["feature"]:<35} {row["mean_abs_shap"]:.5f}')

        # Plot
        fig, ax = plt.subplots(figsize=(9, 6))
        shap_df_plot = shap_df.head(15).sort_values('mean_abs_shap')
        colors = ['#534AB7' if f in EXPECTED_IMPORTANT_FEATURES else '#B4B2A9'
                  for f in shap_df_plot['feature']]
        ax.barh(shap_df_plot['feature'], shap_df_plot['mean_abs_shap'], color=colors)
        ax.set_xlabel('Mean |SHAP value| (weighted avg)', fontsize=11)
        ax.set_title('SHAP Feature Importance — Voting Ensemble (komposit)', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#534AB7', label='Expected important'),
            Patch(facecolor='#B4B2A9', label='Other'),
        ]
        ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
        plt.tight_layout()

        plot_path = os.path.join(OUTPUT_DIR, 'shap_voting_ensemble.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()

        elapsed = time.time() - t_start
        print(f'\n  Plot disimpan : {plot_path}')
        print(f'  Waktu         : {elapsed/60:.1f} menit')

        return {
            'top_features'   : top_features,
            'shap_df'        : shap_df,
            'weighted_shap'  : weighted_shap,
            'plot_path'      : plot_path,
        }

print('OK EnsembleSHAPAnalyzer terdefinisi.')
print('\nOK Semua class terdefinisi. Siap eksekusi.')

OK EnsembleSHAPAnalyzer terdefinisi.

OK Semua class terdefinisi. Siap eksekusi.


---
## Main Loop — Evaluasi D1–D4 per Kandidat

In [12]:
gate2_results = []

for cand in CANDIDATES:
    key   = f"{cand['model_name']}__{cand['balance']}"
    model = tuned_models.get(key)
    if model is None:
        print(f'SKIP {key} — model tidak ditemukan')
        continue

    evaluator = XAIGate2Evaluator(
        model_key     = key,
        model         = model,
        X_shap        = X_shap,
        y_shap        = y_shap,
        X_pi          = X_pi,
        y_pi          = y_pi,
        feature_names = feature_names,
    )
    result = evaluator.run()
    result['tuned_pr_auc'] = cand['tuned_pr_auc']
    gate2_results.append(result)

    # Log ke WandB
    run = wandb.init(
        project = WANDB_PROJECT,
        group   = WANDB_GATE2_GROUP,
        name    = key,
        config  = {
            'model_key'   : key,
            'expected_features': EXPECTED_IMPORTANT_FEATURES,
            'xai_top_n'   : XAI_TOP_N,
            'min_overlap' : XAI_MIN_OVERLAP,
        },
        reinit='finish_previous',
    )
    wandb.log({
        'val_pr_auc'   : cand['tuned_pr_auc'],
        'd1_pass'      : int(result['d1_pass']),
        'd1_overlap'   : result['d1_overlap'],
        'd2_pass'      : int(result['d2_pass']),
        'd2_overlap'   : result['d2_overlap'],
        'd3_pass'      : int(result['d3_pass']),
        'd4_pass'      : int(result['d4_pass']),
        'all_pass'     : int(result['all_pass']),
    })
    if os.path.exists(result['shap_plot']):
        wandb.log({'shap_plot': wandb.Image(result['shap_plot'])})
    wandb.finish()

print('\n' + '=' * 65)
print('  EVALUASI INDIVIDUAL SELESAI')
print('=' * 65)


  XAI Gate #2: lightgbm__class_weight

  [D1] SHAP consistency

  SHAP: lightgbm__class_weight
  Top-10 SHAP features:
       1. Contract_Two year                   1.00496
       2. PaymentMethod_Electronic check      0.57558
       3. InternetService_Fiber optic         0.51315
    *  4. tenure                              0.45957
       5. Contract_One year                   0.29630
    *  6. MonthlyCharges                      0.27149
       7. PaperlessBilling                    0.23097
       8. monthly_to_total_ratio              0.21348
       9. MultipleLines                       0.19538
      10. StreamingTV                         0.13036
  Plot disimpan : /kaggle/working/artifacts/shap_lightgbm__class_weight.png
  Waktu         : 3.9 detik
  D1 SHAP overlap: 4/5 (80%) — LULUS
    [OK] Contract                  via ['Contract_Two year', 'Contract_One year']
    [OK] tenure                    via ['tenure']
    [OK] MonthlyCharges            via ['MonthlyCharges']
    [--] 

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260407_080638-mfmvitgh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lightgbm__class_weight
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/mfmvitgh
wandb: updating run metadata; uploading summary
wandb: uploading history steps 0-1, summary
wandb: 
wandb: Run history:
wandb:   all_pass ▁
wandb: d1_overlap ▁
wandb:    d1_pass ▁
wandb: d2_overlap ▁
wandb:    d2_pass ▁
wandb:    d3_pass ▁
wandb:    d4_pass ▁
wandb: val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb:   all_pass 1
wandb: d1_overlap 0.8
wandb:    d1_pass 1
wandb: d2_overlap 1
wandb:    d2_pass 1
wandb:    d3_pass 1
wandb:    d4_pass 1
wandb: val_pr_auc 0.7522
wandb: 
wandb: 🚀 View run lightgbm__class_weight at: https://wandb.ai/ardiyan


  XAI Gate #2: xgboost__class_weight

  [D1] SHAP consistency

  SHAP: xgboost__class_weight
  Top-10 SHAP features:
       1. Contract_Two year                   0.84708
       2. PaymentMethod_Electronic check      0.58974
    *  3. tenure                              0.52004
       4. InternetService_Fiber optic         0.43576
    *  5. MonthlyCharges                      0.35078
       6. Contract_One year                   0.27778
       7. monthly_to_total_ratio              0.21037
       8. PaperlessBilling                    0.16815
       9. MultipleLines                       0.15104
    * 10. tc_residual                         0.13579
  Plot disimpan : /kaggle/working/artifacts/shap_xgboost__class_weight.png
  Waktu         : 41.9 detik
  D1 SHAP overlap: 5/5 (100%) — LULUS
    [OK] Contract                  via ['Contract_Two year', 'Contract_One year']
    [OK] tenure                    via ['tenure']
    [OK] MonthlyCharges            via ['MonthlyCharges']
    [OK] t

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260407_080732-rz0e0uyh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run xgboost__class_weight
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/rz0e0uyh
wandb: updating run metadata; uploading summary
wandb: uploading media/images/shap_plot_1_626a55fd77c7ff7e0b73.png; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt
wandb: 
wandb: Run history:
wandb:   all_pass ▁
wandb: d1_overlap ▁
wandb:    d1_pass ▁
wandb: d2_overlap ▁
wandb:    d2_pass ▁
wandb:    d3_pass ▁
wandb:    d4_pass ▁
wandb: val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb:   all_pass 1
wandb: d1_overlap 1
wandb:    d1_pass 1
wandb: d2_overlap 1
wandb:    d2_pass 1
wandb:    


  XAI Gate #2: xgboost__smote

  [D1] SHAP consistency

  SHAP: xgboost__smote
  Top-10 SHAP features:
       1. Contract_Two year                   0.72263
    *  2. tenure                              0.54590
       3. PaymentMethod_Electronic check      0.53894
       4. InternetService_Fiber optic         0.51476
       5. Contract_One year                   0.26908
    *  6. MonthlyCharges                      0.23992
       7. PaperlessBilling                    0.23028
       8. monthly_to_total_ratio              0.21572
       9. MultipleLines                       0.18801
      10. OnlineSecurity                      0.14968
  Plot disimpan : /kaggle/working/artifacts/shap_xgboost__smote.png
  Waktu         : 42.7 detik
  D1 SHAP overlap: 4/5 (80%) — LULUS
    [OK] Contract                  via ['Contract_Two year', 'Contract_One year']
    [OK] tenure                    via ['tenure']
    [OK] MonthlyCharges            via ['MonthlyCharges']
    [--] tc_residual            

...


  XAI Gate #2: logistic_regression__smote

  [D1] SHAP consistency

  SHAP: logistic_regression__smote
  Top-10 SHAP features:
       1. Contract_Two year                   0.77703
       2. InternetService_Fiber optic         0.50309
    *  3. tenure                              0.37450
       4. StreamingTV                         0.32886
       5. PaymentMethod_Electronic check      0.32238
       6. StreamingMovies                     0.31481
       7. Contract_One year                   0.30690
       8. service_count                       0.25237
       9. tenure_group_G4_44_72               0.23562
      10. PaperlessBilling                    0.23079
  Plot disimpan : /kaggle/working/artifacts/shap_logistic_regression__smote.png
  Waktu         : 0.5 detik
  D1 SHAP overlap: 3/5 (60%) — LULUS
    [OK] Contract                  via ['Contract_Two year', 'Contract_One year']
    [OK] tenure                    via ['tenure', 'tenure_group_G4_44_72']
    [--] MonthlyCharges       

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has fe

selesai (0.4s)
    Top-10: ['Contract_Two year', 'InternetService_Fiber optic', 'monthly_to_total_ratio', 'StreamingMovies', 'StreamingTV', 'tenure', 'PaymentMethod_Electronic check', 'Contract_One year', 'MultipleLines', 'MonthlyCharges']
  D2 PI overlap: 4/5 (80%) — LULUS
    [OK] Contract                  via ['Contract_Two year', 'Contract_One year']
    [OK] tenure                    via ['tenure']
    [OK] MonthlyCharges            via ['MonthlyCharges']
    [--] tc_residual               tidak ditemukan di top-10
    [OK] InternetService           via ['InternetService_Fiber optic']

  [D3] Garbage feature check
  D3 LULUS: tidak ada garbage feature di top-5

  [D4] Direction check
    [OK] MonthlyCharges                 expected=+  actual=+  corr=+1.0000
    [OK] tenure                         expected=-  actual=-  corr=-1.0000
    [OK] tc_residual                    expected=+  actual=+  corr=+1.0000
    D4 result: 3/3 fitur arah benar → LULUS

  ── VERDICT: logistic_regressio

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260407_080909-g4s05ehb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run logistic_regression__smote
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/g4s05ehb
wandb: updating run metadata; uploading summary
wandb: uploading history steps 0-1, summary
wandb: 
wandb: Run history:
wandb:   all_pass ▁
wandb: d1_overlap ▁
wandb:    d1_pass ▁
wandb: d2_overlap ▁
wandb:    d2_pass ▁
wandb:    d3_pass ▁
wandb:    d4_pass ▁
wandb: val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb:   all_pass 1
wandb: d1_overlap 0.6
wandb:    d1_pass 1
wandb: d2_overlap 0.8
wandb:    d2_pass 1
wandb:    d3_pass 1
wandb:    d4_pass 1
wandb: val_pr_auc 0.7406
wandb: 
wandb: 🚀 View run logistic_regression__smote at: https://wandb.


  EVALUASI INDIVIDUAL SELESAI


---
## SHAP Penuh — Voting Ensemble

In [13]:
ens_shap_result = None

if voting_ensemble is not None:
    ens_analyzer  = EnsembleSHAPAnalyzer(
        ensemble      = voting_ensemble,
        X_sample      = X_ens,
        feature_names = feature_names,
    )
    ens_shap_result = ens_analyzer.run()

    # Evaluasi overlap D1 untuk ensemble
    top_ens  = ens_shap_result['top_features']
    # Prefix-based matching (konsisten dengan XAIGate2Evaluator._evaluate_overlap)
    found = []
    for expected in EXPECTED_IMPORTANT_FEATURES:
        if any(f == expected or f.startswith(expected + '_') for f in top_ens):
            found.append(expected)
    overlap  = len(found) / len(EXPECTED_IMPORTANT_FEATURES)
    ens_d1   = overlap >= XAI_MIN_OVERLAP

    print(f'\n  Ensemble D1 overlap: {len(found)}/{len(EXPECTED_IMPORTANT_FEATURES)} '
          f'({overlap:.0%}) — {"LULUS" if ens_d1 else "GAGAL"}')

    # Evaluasi D4 untuk ensemble
    ens_d4_checker = DirectionChecker(
        model_key      = 'voting_ensemble',
        X_sample       = X_ens,
        shap_values_sv = ens_shap_result['weighted_shap'],
        feature_names  = feature_names,
    )
    ens_d4_result = ens_d4_checker.run()

    # Log ensemble ke WandB
    run = wandb.init(
        project = WANDB_PROJECT,
        group   = WANDB_GATE2_GROUP,
        name    = 'voting_ensemble',
        config  = {'model_key': 'voting_ensemble'},
        reinit  = 'finish_previous',
    )
    wandb.log({
        'val_pr_auc': 0.7525,
        'd1_overlap': overlap,
        'd1_pass'   : int(ens_d1),
        'd4_pass'   : int(ens_d4_result['d4_pass']),
    })
    if os.path.exists(ens_shap_result['plot_path']):
        wandb.log({'shap_plot': wandb.Image(ens_shap_result['plot_path'])})
    wandb.finish()

    ens_gate2 = {
        'key'          : 'voting_ensemble',
        'tuned_pr_auc' : 0.7525,
        'd1_pass'      : ens_d1,
        'd1_overlap'   : overlap,
        'd1_found'     : found,
        'd2_pass'      : None,    # PI tidak dijalankan untuk ensemble (terlalu mahal)
        'd3_pass'      : True,    # Ensemble tidak punya garbage feature by construction
        'd4_pass'      : ens_d4_result['d4_pass'],
        'd4_details'   : ens_d4_result['direction_results'],
        'all_pass'     : ens_d1 and ens_d4_result['d4_pass'],
        'shap_top10'   : top_ens,
        'shap_plot'    : ens_shap_result['plot_path'],
        'note'         : 'D2 (PI) tidak dijalankan untuk ensemble — terlalu mahal komputasinya',
    }
else:
    print('SKIP voting_ensemble — tidak ditemukan')
    ens_gate2 = None


  SHAP Ensemble (komposit)
  Komponen: ['lightgbm_class_weight', 'xgboost_class_weight', 'xgboost_smote']
  Bobot   : [5, 3, 1]
  Menghitung SHAP untuk lightgbm_class_weight (bobot=5)...
  Menghitung SHAP untuk xgboost_class_weight (bobot=3)...
  Menghitung SHAP untuk xgboost_smote (bobot=1)...

  Top-10 SHAP Ensemble:
       1. Contract_Two year                   0.91276
       2. PaymentMethod_Electronic check      0.57096
       3. InternetService_Fiber optic         0.48192
    *  4. tenure                              0.47135
    *  5. MonthlyCharges                      0.28204
       6. Contract_One year                   0.28121
       7. monthly_to_total_ratio              0.21164
       8. PaperlessBilling                    0.20478
       9. MultipleLines                       0.17136
    * 10. tc_residual                         0.12577

  Plot disimpan : /kaggle/working/artifacts/shap_voting_ensemble.png
  Waktu         : 0.6 menit

  Ensemble D1 overlap: 5/5 (100%) — LUL

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260407_080945-wqhkvwwj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run voting_ensemble
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/wqhkvwwj
wandb: updating run metadata; uploading summary
wandb: uploading media/images/shap_plot_1_4c79a81c2b412da1e54b.png
wandb: 
wandb: Run history:
wandb: d1_overlap ▁
wandb:    d1_pass ▁
wandb:    d4_pass ▁
wandb: val_pr_auc ▁
wandb: 
wandb: Run summary:
wandb: d1_overlap 1
wandb:    d1_pass 1
wandb:    d4_pass 1
wandb: val_pr_auc 0.7525
wandb: 
wandb: 🚀 View run voting_ensemble at: https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/wqhkvwwj
wandb: ⭐️ View project at: https://wandb.ai/ardiyanto24-indonesian-national-polic

---
## Ringkasan Verdict

In [14]:
SEP = '=' * 65
print(SEP)
print('  XAI GATE #2 — RINGKASAN VERDICT')
print(SEP)
print(f'  {"Model":<45} {"D1":>4} {"D2":>4} {"D3":>4} {"D4":>4}  {"Verdict"}')
print(f'  {"-"*45} {"-"*4} {"-"*4} {"-"*4} {"-"*4}  {"-"*8}')

all_results = gate2_results[:]
if ens_gate2:
    all_results.append(ens_gate2)

passed_models = []
failed_models = []

for r in all_results:
    d1 = 'OK' if r.get('d1_pass') else 'FAIL'
    d2 = 'OK' if r.get('d2_pass') else ('N/A' if r.get('d2_pass') is None else 'FAIL')
    d3 = 'OK' if r.get('d3_pass') else 'FAIL'
    d4 = 'OK' if r.get('d4_pass') else 'FAIL'
    verdict = 'LULUS' if r.get('all_pass') else 'GAGAL'
    print(f'  {r["key"]:<45} {d1:>4} {d2:>4} {d3:>4} {d4:>4}  {verdict}')
    if r.get('all_pass'):
        passed_models.append(r['key'])
    else:
        failed_models.append(r['key'])

print()
print(f'  Model LULUS  ({len(passed_models)}): {passed_models}')
print(f'  Model GAGAL  ({len(failed_models)}): {failed_models}')
print()
print(f'  Langkah berikutnya: 07_final_evaluation/')
print(f'    Evaluasi final hanya untuk model yang lulus XAI Gate #2')
print(SEP)

  XAI GATE #2 — RINGKASAN VERDICT
  Model                                           D1   D2   D3   D4  Verdict
  --------------------------------------------- ---- ---- ---- ----  --------
  lightgbm__class_weight                          OK   OK   OK   OK  LULUS
  xgboost__class_weight                           OK   OK   OK   OK  LULUS
  xgboost__smote                                  OK   OK   OK   OK  LULUS
  lightgbm__smote                                 OK   OK   OK   OK  LULUS
  logistic_regression__class_weight               OK   OK   OK   OK  LULUS
  logistic_regression__smote                      OK   OK   OK   OK  LULUS
  voting_ensemble                                 OK  N/A   OK   OK  LULUS

  Model LULUS  (7): ['lightgbm__class_weight', 'xgboost__class_weight', 'xgboost__smote', 'lightgbm__smote', 'logistic_regression__class_weight', 'logistic_regression__smote', 'voting_ensemble']
  Model GAGAL  (0): []

  Langkah berikutnya: 07_final_evaluation/
    Evaluasi final hany

---
## Visualisasi Perbandingan D1–D4

In [15]:
# Heatmap D1–D4 per model
individual_results = [r for r in gate2_results]  # hanya individual (bukan ensemble)

keys      = [r['key'] for r in individual_results]
dim_names = ['D1\n(SHAP)', 'D2\n(PI)', 'D3\n(Garbage)', 'D4\n(Direction)']
dim_keys  = ['d1_pass', 'd2_pass', 'd3_pass', 'd4_pass']

matrix = []
for r in individual_results:
    row = []
    for dk in dim_keys:
        val = r.get(dk)
        row.append(1 if val else 0)
    matrix.append(row)

matrix = np.array(matrix)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(dim_names)))
ax.set_xticklabels(dim_names, fontsize=10)
ax.set_yticks(range(len(keys)))
ax.set_yticklabels(keys, fontsize=9)

for i in range(len(keys)):
    for j in range(len(dim_names)):
        val  = matrix[i, j]
        text = 'LULUS' if val == 1 else 'GAGAL'
        ax.text(j, i, text, ha='center', va='center', fontsize=9,
                color='white' if val == 0 else '#0F6E56', fontweight='bold')

ax.set_title('XAI Gate #2 — Heatmap D1–D4 per Kandidat', fontsize=12, fontweight='bold')
plt.tight_layout()

heatmap_path = os.path.join(OUTPUT_DIR, 'xai_gate2_heatmap.png')
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Heatmap disimpan: {heatmap_path}')

Heatmap disimpan: /kaggle/working/artifacts/xai_gate2_heatmap.png


---
## Simpan Output untuk Notebook 07

In [16]:
def make_serializable(obj):
    """Konversi numpy types ke Python native untuk JSON."""
    if isinstance(obj, (np.bool_,)):    return bool(obj)   # harus pertama: NumPy >= 2.0
    if isinstance(obj, bool):           return bool(obj)
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):           return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):           return [make_serializable(i) for i in obj]
    return obj
    
# Susun output untuk Notebook 07
# Sertakan semua kandidat — Notebook 07 filter sendiri berdasarkan all_pass
serializable_results = []
for r in all_results:
    r_clean = {k: v for k, v in r.items()
               if k not in ('shap_df', 'pi_df', 'shap_values_sv', 'weighted_shap')}
    serializable_results.append(make_serializable(r_clean))

output = {
    'notebook'           : 'Notebook 06 — XAI Gate #2',
    'expected_features'  : EXPECTED_IMPORTANT_FEATURES,
    'xai_top_n'          : XAI_TOP_N,
    'xai_min_overlap'    : XAI_MIN_OVERLAP,
    'gate2_results'      : serializable_results,
    'passed_models'      : passed_models,
    'failed_models'      : failed_models,
}

output_path = os.path.join(OUTPUT_DIR, 'xai_gate2_results.json')
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'OK xai_gate2_results.json disimpan: {output_path}')
print(json.dumps({
    'total_evaluated': len(all_results),
    'passed'         : passed_models,
    'failed'         : failed_models,
}, indent=2))

OK xai_gate2_results.json disimpan: /kaggle/working/artifacts/xai_gate2_results.json
{
  "total_evaluated": 7,
  "passed": [
    "lightgbm__class_weight",
    "xgboost__class_weight",
    "xgboost__smote",
    "lightgbm__smote",
    "logistic_regression__class_weight",
    "logistic_regression__smote",
    "voting_ensemble"
  ],
  "failed": []
}


---
## Ringkasan Final

In [17]:
import glob

SEP = '=' * 65
print(SEP)
print('  NOTEBOOK 06 SELESAI')
print(SEP)
print()
print(f'  Total kandidat dievaluasi : {len(all_results)}')
print(f'  Model lulus XAI Gate #2   : {len(passed_models)}')
print(f'  Model gagal               : {len(failed_models)}')
print()
print(f'  Passed : {passed_models}')
print(f'  Failed : {failed_models}')
print()
print('  Artifacts:')
for fpath in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*'))):
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {os.path.basename(fpath):<55} ({size_kb:.0f} KB)')
print()
print('  Langkah berikutnya: 07_final_evaluation/')
print('    Load xai_gate2_results.json')
print('    Evaluasi final hanya model yang all_pass=True')
print('    Pilih model final untuk deployment')
print(SEP)

  NOTEBOOK 06 SELESAI

  Total kandidat dievaluasi : 7
  Model lulus XAI Gate #2   : 7
  Model gagal               : 0

  Passed : ['lightgbm__class_weight', 'xgboost__class_weight', 'xgboost__smote', 'lightgbm__smote', 'logistic_regression__class_weight', 'logistic_regression__smote', 'voting_ensemble']
  Failed : []

  Artifacts:
  shap_lightgbm__class_weight.png                         (79 KB)
  shap_lightgbm__smote.png                                (78 KB)
  shap_logistic_regression__class_weight.png              (87 KB)
  shap_logistic_regression__smote.png                     (86 KB)
  shap_voting_ensemble.png                                (84 KB)
  shap_xgboost__class_weight.png                          (82 KB)
  shap_xgboost__smote.png                                 (79 KB)
  xai_gate2_heatmap.png                                   (51 KB)
  xai_gate2_results.json                                  (11 KB)

  Langkah berikutnya: 07_final_evaluation/
    Load xai_gate2_results.j